# 🧪 模块: Python 数据清洗进阶 — 笔试/面试 20题精选版

本练习遵循标准化教学工作流，专为 Senior DA 笔试/面试拔高设计。涵盖：复杂提取、高级分组、重塑、时序与多条件判别。

## 模块 0: 函数加油站 (Function Cheat Sheet)
在我们开始前，先复习一下进阶清洗的核心招式：
- **`str.extract()`**: 正则提取 (需加括号分组)。SQL类比: `REGEXP_EXTRACT()`
- **`.transform()`**: 组内变换，返回与原表等长的数据。SQL类比: 窗口函数 `SUM() OVER(PARTITION BY)`
- **`np.select()`**: 多条件分支判断。SQL类比: `CASE WHEN ... THEN ... END`
- **`.clip()`**: 极值截断，替代繁琐的 if-else 盖帽法。SQL类比: `LEAST(GREATEST(x, min), max)`
- **`.melt()`**: 宽表变长表 (逆透视)。SQL类比: `UNPIVOT`


## 模块 1: 概念映射
进阶数据处理不仅仅是写代码，更是对数据颗粒度（粒度）的变形。你需要像捏橡皮泥一样熟练地对 DataFrame 聚合 (`agg`)、打散 (`explode`)、拉长 (`melt`)。

## 模块 2: 数据准备
*(注: 本节针对具体的高频小场景边界测试（Edge Cases），因此使用精心设计的验证集，以覆盖各种容易引发报错的脏数据格式。)*

## 模块 3: 分级挑战 (20 个魔鬼细节测试)

## 模块 4: 参考答案 (请在独立练习后查看)

### Part 1: 复杂字符串处理 (Advanced String Operations)

**Q1: 提取金额**
提取文本中的数字并转为 float。注意有 `K` (千) 的情况。

`text = ['Price: $1.2K', 'Just $500', 'Cost is $2.5K', 'Free']`

要求输出: `[1200.0, 500.0, 2500.0, 0.0]`

In [ ]:
df = pd.DataFrame({'text': ['Price: $1.2K', 'Just $500', 'Cost is $2.5K', 'Free']})
# 你的代码


In [ ]:
# ✅ Q1 参考答案
import re

def parse_money(x):
    if 'Free' in x:
        return 0.0
    num = float(re.search(r'[\d\.]+', x).group())
    if 'K' in x:
        num *= 1000
    return num

df['amount'] = df['text'].apply(parse_money)
print(df['amount'].tolist())

**Q2: 提取邮箱后缀 (正则表达式提取)**
提取所有邮箱的后缀 (例如 `@gmail.com` 提取 `gmail`)。

`emails = ['user@gmail.com', 'admin@163.com', 'test@yahoo.com']`

要求输出: `['gmail', '163', 'yahoo']`

In [ ]:
import pandas as pd
df = pd.DataFrame({'email': ['user@gmail.com', 'admin@163.com', 'test@yahoo.com']})
# 你的代码


In [ ]:
# ✅ Q2 参考答案
# 💡 .str.extract 需要配合正则表达式使用，并且必须要有 () 分组
df['domain'] = df['email'].str.extract(r'@(.*)\.com')
print(df['domain'].tolist())

**Q3: 拆分姓名列**
将 `name` 列拆分为 `first_name` 和 `last_name` 两列。

`names = ['John Doe', 'Alice Smith', 'Bob']`

要求: 新增两列，Bob 的 last_name 为 None

In [ ]:
df = pd.DataFrame({'name': ['John Doe', 'Alice Smith', 'Bob']})
# 你的代码


In [ ]:
# ✅ Q3 参考答案
# 💡 expand=True 可以直接把拆分结果变成多列 DataFrame
df[['first_name', 'last_name']] = df['name'].str.split(' ', n=1, expand=True)
print(df)

### Part 2: 神奇的 GroupBy (Advanced GroupBy)

**Q4: 分组填充空值 (Group-specific Imputation)**
根据不同城市的平均分，来填充该城市的空值。

要求: BJ 的 NaN 填 80，SH 的 NaN 填 90

In [ ]:
import numpy as np
df = pd.DataFrame({'city': ['BJ', 'BJ', 'SH', 'SH'], 'score': [80, np.nan, 90, np.nan]})
# 你的代码


In [ ]:
# ✅ Q4 参考答案
# 💡 transform 绝配！计算每组的均值，并且保持长度和原表一样，直接用 fillna 接收
df['score'] = df['score'].fillna(df.groupby('city')['score'].transform('mean'))
print(df)

**Q5: 组内占比计算**
计算每个员工的销售额在所在部门的占比。

In [ ]:
df = pd.DataFrame({'dept': ['A', 'A', 'B', 'B'], 'name': ['P1', 'P2', 'P3', 'P4'], 'sales': [100, 300, 200, 200]})
# 你的代码


In [ ]:
# ✅ Q5 参考答案
# 💡 同样是 transform 广播组总和到每一行
df['ratio'] = df['sales'] / df.groupby('dept')['sales'].transform('sum')
print(df['ratio'].tolist())

**Q6: 分组过滤 (GroupBy Filter)**
找出平均分及格 (>=60) 的全体班级成员！(注意是找出成员，不是找出班级)

要求输出: 只保留 class 1 的两行记录

In [ ]:
df = pd.DataFrame({'class': [1, 1, 2, 2], 'score': [70, 80, 30, 50]})
# 你的代码


In [ ]:
# ✅ Q6 参考答案
# 💡 filter 是以组为单位返回 True/False，如果为 False 则扔掉该组所有行
df_filtered = df.groupby('class').filter(lambda x: x['score'].mean() >= 60)
print(df_filtered)

**Q7: 高阶聚合并重命名 (Named Aggregation)**
按部门分组，不仅要算平均年龄，还要算最高工资，还要给这两列起好名字 (`avg_age`, `max_salary`)。

要求: 一步写法

In [ ]:
df = pd.DataFrame({'dept': ['A','A','B','B'], 'age': [25,35,40,50], 'salary': [5000, 7000, 8000, 10000]})
# 你的代码


In [ ]:
# ✅ Q7 参考答案
# 💡 Pandas 原生的 Named Aggregation（无需 MultiIndex 打平）
result = df.groupby('dept').agg(
    avg_age=('age', 'mean'),
    max_salary=('salary', 'max')
)
print(result)

### Part 3: 窗口与序列 (Lag & Rolling)

**Q8: 寻找状态突变**
找出用户的VIP等级发生变化的行（即今天等级和昨天不一样）。

要求输出: 只提取状态突变的行

In [ ]:
df = pd.DataFrame({
    'uid': [1, 1, 1, 2, 2],
    'date': ['01-01', '01-02', '01-03', '01-01', '01-02'],
    'level': ['V1', 'V1', 'V2', 'V1', 'V1']
})
# 你的代码


In [ ]:
# ✅ Q8 参考答案
# 💡 利用 shift 获取拉链数据比对，注意一定要加上 groupby('uid')，否则会和上个人的数据比对！
shifted_level = df.groupby('uid')['level'].shift(1)
df['is_changed'] = (df['level'] != shifted_level) & (shifted_level.notna())
print(df[df['is_changed']])

**Q9: 组内累计求和 (Cumulative Sum)**
计算每个用户的累计消费金额（按日期排列）。

要求新增一列 `cum_amount`，对于 uid 1 分别是 100, 150, 170

In [ ]:
df = pd.DataFrame({'uid': [1,1,1,2,2], 'amount': [100,50,20,200,100]})
# 你的代码


In [ ]:
# ✅ Q9 参考答案
# 💡 cumsum 原带原生组内累加特性
df['cum_amount'] = df.groupby('uid')['amount'].cumsum()
print(df['cum_amount'].tolist())

**Q10: 组内排名计算**
计算每个班级里，学生按分数的排名（分数高的排第1）。

In [ ]:
df = pd.DataFrame({'class': ['A', 'A', 'A', 'B', 'B'], 'score': [90, 80, 95, 70, 80]})
# 你的代码


In [ ]:
# ✅ Q10 参考答案
# 💡 rank 方法配合 ascending=False
df['rank'] = df.groupby('class')['score'].rank(ascending=False, method='min')
print(df['rank'].tolist())

### Part 4: 长宽表转换 (Reshaping: Melt & Pivot)

**Q11: 宽表变长表 (Melt)**
这是典型的 Excel 透视后的宽表，你需要把它逆透视回长表。

要求变成: `Name`, `Subject`, `Score`

In [ ]:
df = pd.DataFrame({'Name': ['Tom', 'Jerry'], 'Math': [90, 85], 'English': [80, 95]})
# 你的代码


In [ ]:
# ✅ Q11 参考答案
# 💡 id_vars 是保留不动的列名，var_name 是你要把旧列名塞进去的新列，value_name 是数值列
long_df = df.melt(id_vars='Name', var_name='Subject', value_name='Score')
print(long_df)

**Q12: 长表变宽表 (Pivot / Pivot_table)**
把 Q11 刚才生成的 long_df 再变回宽表！并处理如果有重复数据则取平均值。

In [ ]:
long_df = pd.DataFrame({
    'Name': ['Tom', 'Tom', 'Jerry', 'Jerry', 'Tom'],
    'Subject': ['Math', 'English', 'Math', 'English', 'Math'],
    'Score': [90, 80, 85, 95, 100]  # Tom 的 Math 有两次记录 90 和 100
})
# 你的代码


In [ ]:
# ✅ Q12 参考答案
# 💡 遇到重复数据必须用 pivot_table (不能用 pivot)，它自带 aggfunc='mean'
wide_df = long_df.pivot_table(index='Name', columns='Subject', values='Score', aggfunc='mean').reset_index()
print(wide_df)

### Part 5: 条件判断与掩码 (Conditionals & Masking)

**Q13: 多条件复杂派生列 (np.select 进阶)**
根据用户的注册天数 `tenure` 和消费金额 `amount` 划分层级：
- `tenure > 365` 且 `amount > 1000` -> 'Core'
- `tenure > 365` 且 `amount <= 1000` -> 'Loyal'
- `tenure <= 365` 且 `amount > 1000` -> 'Whale'
- 其他 -> 'Newbie'

In [ ]:
df = pd.DataFrame({'tenure': [400, 500, 100, 50], 'amount': [1500, 500, 2000, 100]})
# 你的代码


In [ ]:
# ✅ Q13 参考答案
import numpy as np
# 💡 np.select 是处理多维度 if-elif 逻辑的最佳实践
conds = [
    (df['tenure']>365) & (df['amount']>1000),
    (df['tenure']>365) & (df['amount']<=1000),
    (df['tenure']<=365) & (df['amount']>1000)
]
labels = ['Core', 'Loyal', 'Whale']
df['Segment'] = np.select(conds, labels, default='Newbie')
print(df['Segment'].tolist())

**Q14: 截断极端值 (Outlier Capping)**
找出 df['salary'] 的 95% 分位数和 5% 分位数。将大于 95% 分位数的值强行截断为 95% 分位数的值，小于 5% 的同理。
*(不用 drop，用数值替换)*

In [ ]:
df = pd.DataFrame({'salary': [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 1000000]})
# 你的代码


In [ ]:
# ✅ Q14 参考答案
# 💡 Pandas 自带 clip 函数，完美解决盖帽法！
upper = df['salary'].quantile(0.95)
lower = df['salary'].quantile(0.05)
df['salary_capped'] = df['salary'].clip(lower=lower, upper=upper)
print(df['salary_capped'].tolist())

### Part 6: 日期专精 (Datetime Hero)

**Q15: 脏日期解析**
解析带有脏数据的日期，如果解析失败则变为 `NaT` (不要报错！)。

In [ ]:
df = pd.DataFrame({'date': ['2023-01-01', '2023-13-45', '2023-01-03']})
# 你的代码


In [ ]:
# ✅ Q15 参考答案
# 💡 to_datetime 的 errors='coerce' 是保命参数
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(df['date'].tolist())

**Q16: 计算两个日期的差集 (天数)**
计算下单日期 (`order_date`) 到收货日期 (`delivery_date`) 花了几天？

In [ ]:
df = pd.DataFrame({'order_date': ['2023-01-01', '2023-01-05'], 'delivery_date': ['2023-01-10', '2023-01-07']})
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])
# 你的代码


In [ ]:
# ✅ Q16 参考答案
# 💡 日期相减得到的是 Timedelta 对象，用 .dt.days 提取纯数字
df['days_taken'] = (df['delivery_date'] - df['order_date']).dt.days
print(df['days_taken'].tolist())

**Q17: 提取月份首末日特征**
判断这个事件是否发生在每个月的月底？(如果是月底最后一天，返回 True，否则 False)

In [ ]:
df = pd.DataFrame({'date': ['2023-01-31', '2023-02-15', '2023-02-28']})
df['date'] = pd.to_datetime(df['date'])
# 你的代码


In [ ]:
# ✅ Q17 参考答案
# 💡 .dt.is_month_end 直接提供这个属性！
df['is_month_end'] = df['date'].dt.is_month_end
print(df['is_month_end'].tolist())

### Part 7: 结构重组与字典映射 (Remap & Explode)

**Q18: `.map()` 与 `.replace()` 差异对比**
用一个字典将城市缩写映射为全拼。

若用 `map()` 遇到找不到的词会发生什么？安全的替换怎么写？

In [ ]:
df = pd.DataFrame({'city': ['BJ', 'SH', 'GZ']})
mapping = {'BJ': 'Beijing', 'SH': 'Shanghai'}
# 分别用 map 和 replace 尝试一下


In [ ]:
# ✅ Q18 参考答案
# 💡 map 会把字典里没有的词变成 NaN！replace 则是维持原样。
print('Map 结果 (GZ 变 NaN):\n', df['city'].map(mapping))
print('Replace 结果 (GZ 维持原样):\n', df['city'].replace(mapping))

**Q19: 把列表炸裂开 (Explode)**
在处理 JSON 数据时，某个字段是一箩筐标签的列表。你需要把它炸成多行！

In [ ]:
df = pd.DataFrame({'uid': [1, 2], 'tags': [['A', 'B'], ['C']]})
# 你的代码


In [ ]:
# ✅ Q19 参考答案
# 💡 explode 是专门用来把单元格里的 List 展开成多行的神器
df_exploded = df.explode('tags')
print(df_exploded)

### Part 8: 坑点和盲区 (Gotchas)

**Q20: Loc 对齐导致的 NaN 陷阱**
你有两个 Series: s1 (索引为 0,1) 和 s2 (索引为 1,2)。你想把他们首尾相加，但不希望有 NaN 传染！

In [ ]:
s1 = pd.Series([10, 20], index=[0, 1])
s2 = pd.Series([100, 200], index=[1, 2])
# 计算 s1 + s2 试试看，如何让它不会出现 0:NaN 和 2:NaN？


In [ ]:
# ✅ Q20 参考答案
# 💡 s1 + s2 如果 index 不匹配就会变空。使用 .add(fill_value=0) 让缺失一方视为 0！
result = s1.add(s2, fill_value=0)
print(result)